# Assignment 2: Social Media & Network Analysis
## "The impact of AI on Music Sectors (Individual, Communities and Industries)"

Undergraduate (UG) Group 1
- Joshua Steinke (s4091863)
- Paul Venatt (s4089896)
- Putu Adhi Wiguna (s4097286)



In [1]:
# Import packages here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import re
import json
import nltk
import string
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from scripts.YouTubeProcessing import YouTubeProcessing


In [2]:
with open('./data/youtubeFlatComments_AI_on_music.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data['comments'])
df.head()

,video_id,video_title,channel_id,channel_title,video_published_at,video_view_count,video_like_count,video_comment_count,comment_id,parent_comment_id,comment_type,author_display_name,author_channel_id,text,text_original,like_count,published_at,updated_at,reply_to_author_display_name,reply_to_author_channel_id
0,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg,NaN,top_level,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw,Never could have predicted this crossover. Two...,Never could have predicted this crossover. Two...,1441,2026-05-13T15:32:19Z,2026-05-13T15:32:19Z,NaN,NaN
1,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhCufgK72,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@MidnightUnity,UCdisatKsUEaCeliNJ6BdKZg,"I was about to say two of my favourite nerds, ...","I was about to say two of my favourite nerds, ...",28,2026-05-13T15:42:20Z,2026-05-13T15:42:20Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw
2,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhSyhPznS,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@AdımızAdımız,UCRELhN7padwcwT94a0m0tnQ,fr,fr,1,2026-05-13T15:44:32Z,2026-05-13T15:44:32Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw
3,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhd5JSn27,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@mymasmith7848,UC7ps19AcOCVsJB3MzW_qsmQ,"Yeeeeehah, I am so eagerly the next two hours ...","Yeeeeehah, I am so eagerly the next two hours ...",2,2026-05-13T15:46:03Z,2026-05-13T15:46:03Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw
4,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkjpDEB6Po,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@zlquis,UCbhxNik1HRXNuFjBb2nXMvg,Literally my 2 favourite content creators. I'm...,Literally my 2 favourite content creators. I'm...,9,2026-05-13T16:05:11Z,2026-05-13T16:05:11Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw


In [3]:
print(f'Shape: {df.shape}')
df.columns

Shape: (14342, 20)


Index(['video_id', 'video_title', 'channel_id', 'channel_title',
       'video_published_at', 'video_view_count', 'video_like_count',
       'video_comment_count', 'comment_id', 'parent_comment_id',
       'comment_type', 'author_display_name', 'author_channel_id', 'text',
       'text_original', 'like_count', 'published_at', 'updated_at',
       'reply_to_author_display_name', 'reply_to_author_channel_id'],
      dtype='str')

In [4]:
# Get a summary of the dataset
data_summary = {
    'number_of_videos': df['video_id'].nunique(),
    'number_of_comments_and_replies': len(df),
    'number_of_unique_users': df['author_channel_id'].nunique(),
    'number_of_top_level_comments': (df['comment_type'] == 'top_level').sum(),
    'number_of_replies': (df['comment_type'] == 'reply').sum()
}

display(data_summary)

{'number_of_videos': 25,
 'number_of_comments_and_replies': 14342,
 'number_of_unique_users': 9265,
 'number_of_top_level_comments': np.int64(2331),
 'number_of_replies': np.int64(12011)}

In [5]:
df['text'].isna().sum() + (df['text'].astype(str).str.strip() == '').sum()

np.int64(7)

In [6]:
df.isna().sum().sort_values(ascending=False)

reply_to_author_channel_id      2331
reply_to_author_display_name    2331
parent_comment_id               2331
author_display_name                0
updated_at                         0
published_at                       0
like_count                         0
text_original                      0
text                               0
author_channel_id                  0
video_id                           0
video_title                        0
comment_id                         0
video_comment_count                0
video_like_count                   0
video_view_count                   0
video_published_at                 0
channel_title                      0
channel_id                         0
comment_type                       0
dtype: int64

In [7]:
# Using NLTK stopwords
nltk.download('stopwords')
tokeniser = TweetTokenizer()

"""
Using English stopwords and punctuation,
as well as some additional stopwords that don't add much meaning to comments
"""
stopwords = stopwords.words('english') + list(string.punctuation)
additional_stopwords = ['ai', 'music', 'song', 'songs', 'video', 'like', 'just', 'get', 'would']
stopwords += (additional_stopwords)

stemmer = PorterStemmer()
processor = YouTubeProcessing(tokeniser=tokeniser, stopwords=stopwords, stemmer=stemmer)

[nltk_data] Downloading package stopwords to /Users/adhi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
"""
Apply text processing functions from YouTubeProcessing.py to:

1. Clean the raw text
2. Tokenise
3. Join the processed tokens back into a string
"""
df['cleaned_text'] = df['text'].apply(processor.clean_raw_text)
df['tokens'] = df['text'].apply(processor.process)
df['processed_text'] = df['tokens'].apply(processor.joined_tokens)

# Inspect the newly created columns
df[['text', 'cleaned_text', 'tokens', 'processed_text']].head()

,text,cleaned_text,tokens,processed_text
0,Never could have predicted this crossover. Two...,never could have predicted this crossover. two...,"[never, could, predict, crossov, two, favorit,...",never could predict crossov two favorit studi ...
1,"I was about to say two of my favourite nerds, ...","i was about to say two of my favourite nerds, ...","[say, two, favourit, nerd, studi, actual, nice...",say two favourit nerd studi actual nicer way w...
2,fr,fr,[fr],fr
3,"Yeeeeehah, I am so eagerly the next two hours ...","yeeeeehah, i am so eagerly the next two hours ...","[yeeeeehah, eagerli, next, two, hour, fine, li...",yeeeeehah eagerli next two hour fine listen
4,Literally my 2 favourite content creators. I'm...,literally my 2 favourite content creators. i'm...,"[liter, favourit, content, creator, even, go, ...",liter favourit content creator even go see ada...


In [11]:
df['text_length'] = df['cleaned_text'].str.len()
df['token_count'] = df['tokens'].apply(len)

df[['text_length', 'token_count']].describe()

,text_length,token_count
count,14342.000000,14342.000000
mean,173.134221,14.858039
std,282.165460,23.987252
min,0.000000,0.000000
25%,34.000000,3.000000
50%,78.500000,7.000000
75%,189.000000,16.000000
max,6192.000000,502.000000


In [12]:
# Remove very short comments
df = df[df['token_count'] >= 2]

In [ ]:
# Save the cleaned comments dataset into a CSV and JSON file
df.to_csv('clean_comments.csv', index=False)
df.to_json('clean_comments.json', orient='records', force_ascii=False, indent=2)

### References
